# Deepfake Deterministic Calibration Training
### Kaggle · Human Faces Dataset → Deterministic CV Features → Logistic Regression Calibration

This notebook:
1. Downloads [`kaustubhdhote/human-faces-dataset`](https://www.kaggle.com/datasets/kaustubhdhote/human-faces-dataset) from Kaggle
2. Runs the **10-feature deterministic forensic pipeline** (Laplacian, Sobel, DFT, FFT, Histogram, Skin HSV, Noise Residual, JPEG Blocking, Edge Ringing, Texture Inconsistency) across 4 image variants per sample
3. Trains a **logistic regression calibration model** on those features
4. Saves `deterministic_calibration.json` — drop it next to `deterministic_analysis.py` and point `DETERMINISTIC_CALIBRATION_PATH` at it

**Works in Google Colab and locally in VS Code.**

In [ ]:
import sys, os
from pathlib import Path

# Detect runtime
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Runtime: {'Google Colab' if IN_COLAB else 'Local / VS Code'}")

# Configuration
KAGGLE_DATASET = "kaustubhdhote/human-faces-dataset"

# Folder-name keywords used to infer binary labels (case-insensitive)
REAL_FOLDER_KEYWORDS = ["real", "genuine", "authentic", "original"]
FAKE_FOLDER_KEYWORDS = ["fake", "ai", "generated", "deepfake", "synthetic", "artificial"]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
MAX_PER_CLASS = None   # integer to cap samples per class, None = all
MAX_WORKERS = 4        # parallel workers for feature extraction
VALIDATION_FRACTION = 0.20
VALIDATION_FRAC = VALIDATION_FRACTION  # backward-compatible alias
EPOCHS = 1200
SEED = 42

ANALYSIS_PROFILE = "fast"
PROFILE_MAX_SIDE_DEFAULTS = {"fast": 512, "balanced": 640, "full": 768}
ANALYSIS_MAX_SIDE = PROFILE_MAX_SIDE_DEFAULTS[ANALYSIS_PROFILE]

# Paths (resolved per runtime)
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "human-faces-dataset"
OUTPUT_DIR = BASE_DIR / "outputs"
CACHE_PATH = OUTPUT_DIR / f"features_{ANALYSIS_PROFILE}_{ANALYSIS_MAX_SIDE}.npz"
CALIB_PATH = OUTPUT_DIR / "deterministic_calibration.json"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR         : {BASE_DIR}")
print(f"DATA_DIR         : {DATA_DIR}")
print(f"OUTPUT_DIR       : {OUTPUT_DIR}")
print(f"ANALYSIS_PROFILE : {ANALYSIS_PROFILE}")
print(f"ANALYSIS_MAX_SIDE: {ANALYSIS_MAX_SIDE}")
print(f"CACHE_PATH       : {CACHE_PATH}")

In [ ]:
import importlib, subprocess

def _ensure(pip_name, import_name=None):
    mod = import_name or pip_name.split("[")[0].replace("-", "_")
    if importlib.util.find_spec(mod) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_ensure("kaggle")
_ensure("opencv-python-headless", "cv2")
_ensure("tqdm")
_ensure("matplotlib")
_ensure("numpy")
_ensure("Pillow", "PIL")
_ensure("pandas")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from PIL import Image

print("All dependencies ready.")

In [ ]:
import zipfile

def setup_kaggle_credentials():
    """Configure Kaggle credentials. In Colab: prompt file upload. Locally: expects ~/.kaggle/kaggle.json."""
    if IN_COLAB:
        creds = Path("/root/.kaggle/kaggle.json")
        if not creds.exists():
            from google.colab import files
            print("Upload your kaggle.json  —  download it from https://www.kaggle.com/settings (API section)")
            uploaded = files.upload()
            if "kaggle.json" not in uploaded:
                raise FileNotFoundError("kaggle.json not uploaded. Cannot download dataset.")
            creds.parent.mkdir(parents=True, exist_ok=True)
            creds.write_bytes(uploaded["kaggle.json"])
            os.chmod(creds, 0o600)
            print("kaggle.json saved to", creds)
    else:
        local = Path.home() / ".kaggle" / "kaggle.json"
        if not local.exists() and not os.getenv("KAGGLE_USERNAME"):
            raise FileNotFoundError(
                "Kaggle credentials not found.\n"
                "Option A: place kaggle.json at ~/.kaggle/kaggle.json\n"
                "Option B: set env vars KAGGLE_USERNAME and KAGGLE_KEY\n"
                "Download from: https://www.kaggle.com/settings  → API → Create New Token"
            )


def download_dataset():
    marker = DATA_DIR / ".downloaded"
    if marker.exists():
        print(f"Dataset already present at {DATA_DIR}  (delete .downloaded to re-download)")
        return

    import kaggle
    print(f"Downloading {KAGGLE_DATASET}  …")
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(
        KAGGLE_DATASET,
        path=str(DATA_DIR),
        unzip=False,
        quiet=False,
    )
    marker.touch()
    print("Download complete.")


setup_kaggle_credentials()
download_dataset()

In [ ]:
# ── Optional Google Drive mount (Colab only) ──────────────────────
if IN_COLAB:
    _mount = input("Mount Google Drive to persist calibration output? [y/N]: ").strip().lower()
    if _mount == "y":
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = Path("/content/drive/MyDrive/deepfake_calibration")
        CALIB_PATH = OUTPUT_DIR / "deterministic_calibration.json"
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Outputs → Google Drive: {OUTPUT_DIR}")
    else:
        print("Outputs saved to Colab session storage (temporary).")
else:
    print(f"Outputs → local: {OUTPUT_DIR}")

In [ ]:
def extract_dataset():
    marker = DATA_DIR / ".extracted"
    if marker.exists():
        print(f"Already extracted — skipping  (delete .extracted to redo)")
        return
    zips = list(DATA_DIR.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip file found in {DATA_DIR}")
    for zf in zips:
        print(f"Extracting {zf.name}  …")
        with zipfile.ZipFile(zf, "r") as z:
            z.extractall(DATA_DIR)
    marker.touch()
    print("Extraction complete.")


def infer_label(name: str):
    """Return 0 (real), 1 (fake), or None from a folder/file name."""
    lo = name.lower()
    for kw in REAL_FOLDER_KEYWORDS:
        if kw in lo:
            return 0
    for kw in FAKE_FOLDER_KEYWORDS:
        if kw in lo:
            return 1
    return None


def inventory_images(root: Path) -> pd.DataFrame:
    rows = []
    for p in root.rglob("*"):
        if p.suffix.lower() not in IMAGE_EXTS:
            continue
        label = None
        # Walk ancestor parts from deepest to shallowest for best match
        for part in reversed(p.relative_to(root).parts[:-1]):
            inferred = infer_label(part)
            if inferred is not None:
                label = inferred
                break
        rows.append({
            "path": str(p),
            "folder": p.parent.name,
            "label": label,
            "label_name": {0: "real", 1: "fake"}.get(label, "unknown"),
        })
    return pd.DataFrame(rows)


extract_dataset()
df_all = inventory_images(DATA_DIR)

print(f"Total images : {len(df_all)}")
print("\nBy folder (top 20):")
print(df_all.groupby(["folder", "label_name"]).size().to_string())

df = df_all[df_all["label"].notna()].copy()
df["label"] = df["label"].astype(int)
print(f"\nLabeled images: {len(df)}")
print(df.groupby("label_name").size().rename("count").to_string())

if len(df) == 0:
    print("\n⚠  No labels could be inferred. Check REAL_FOLDER_KEYWORDS / FAKE_FOLDER_KEYWORDS above.")
    print("Folder names found:", sorted(df_all['folder'].unique().tolist()))

In [ ]:
def load_img_rgb(path: str, size=(128, 128)) -> np.ndarray | None:
    try:
        return np.array(Image.open(path).convert("RGB").resize(size))
    except Exception:
        return None


def validate_and_preview(df: pd.DataFrame, n_per_class=4) -> pd.DataFrame:
    valid_paths = []
    for path in tqdm(df["path"], desc="Validating images", unit="img"):
        bgr = cv2.imread(path)
        if bgr is not None and bgr.size > 0:
            valid_paths.append(path)
    valid = df[df["path"].isin(set(valid_paths))].copy()
    print(f"Valid: {len(valid)} / {len(df)}  (removed {len(df)-len(valid)} unreadable)")

    # Grid preview
    classes = sorted(valid["label_name"].unique())
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(n_per_class * 2.5, len(classes) * 2.5))
    if len(classes) == 1:
        axes = [axes]
    for row_idx, cls in enumerate(classes):
        subset = valid[valid["label_name"] == cls].sample(
            min(n_per_class, len(valid[valid["label_name"] == cls])), random_state=SEED
        )
        for col_idx in range(n_per_class):
            ax = axes[row_idx][col_idx]
            ax.axis("off")
            if col_idx < len(subset):
                img = load_img_rgb(subset.iloc[col_idx]["path"])
                if img is not None:
                    ax.imshow(img)
                    ax.set_title(cls, fontsize=8)
    fig.suptitle("Sample images by class", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
    return valid


df = validate_and_preview(df)

# Optional class cap
if MAX_PER_CLASS:
    df = (
        df.groupby("label", group_keys=False)
          .apply(lambda g: g.sample(min(MAX_PER_CLASS, len(g)), random_state=SEED))
          .reset_index(drop=True)
    )
    print(f"After capping at {MAX_PER_CLASS}/class: {len(df)} images  "
          f"({len(df[df.label==0])} real, {len(df[df.label==1])} fake)")

## Deterministic Analysis Engine

The notebook keeps a standalone copy of the deterministic feature extractor so it still runs on Colab without the local project files.  
Training is now pinned to the `fast` profile: 7 selected forensic features, 2 image variants (`original` + `clahe`), and a 512px max image side before analysis.

In [ ]:
# Deterministic analysis engine (mirrors deterministic_analysis.py)
# If the local project file is importable, use it directly; otherwise use this
# self-contained copy so the notebook works standalone on Google Colab.

try:
    import importlib.util

    _spec = importlib.util.find_spec("deterministic_analysis")
    if _spec is None:
        raise ImportError

    from deterministic_analysis import (
        ANALYZERS,
        DEFAULT_WEIGHTS,
        get_profile_feature_names,
        get_profile_max_image_side,
        get_profile_variant_weights,
        resolve_analysis_profile,
        run_multiview_analyzers,
    )

    print("Loaded deterministic_analysis from local project.")
except ImportError:
    print("Local project not found — using embedded engine.")

    from concurrent.futures import ThreadPoolExecutor, as_completed as _as_completed
    from functools import lru_cache
    from typing import Callable, Dict, Iterable, Mapping

    AnalyzerFn = Callable[[np.ndarray], float]
    DEFAULT_ANALYSIS_PROFILE = "fast"

    def _soft_two_tailed_score(value, low, high, outer_width):
        if high <= low:
            return 0.0
        mid = 0.5 * (low + high)
        half_range = 0.5 * (high - low) + 1e-9
        in_band_distance = abs(value - mid) / half_range
        if low <= value <= high:
            return float(np.clip(0.35 * in_band_distance, 0.0, 1.0))
        outside_distance = ((low - value) if value < low else (value - high)) / (outer_width + 1e-9)
        return float(np.clip(0.35 + 0.65 * outside_distance, 0.0, 1.0))

    def _center_crop_region(bgr):
        h, w = bgr.shape[:2]
        my, mx = h // 4, w // 4
        return my, h - my, mx, w - mx

    def _resize_for_analysis(bgr, max_image_side):
        if max_image_side is None or max_image_side <= 0:
            return bgr
        h, w = bgr.shape[:2]
        longest = max(h, w)
        if longest <= max_image_side:
            return bgr
        scale = float(max_image_side) / float(longest)
        resized_w = max(1, int(round(w * scale)))
        resized_h = max(1, int(round(h * scale)))
        return cv2.resize(bgr, (resized_w, resized_h), interpolation=cv2.INTER_AREA)

    @lru_cache(maxsize=16)
    def _high_frequency_mask(shape):
        h, w = shape
        cy, cx = h // 2, w // 2
        y_grid, x_grid = np.ogrid[:h, :w]
        dist = np.sqrt((x_grid - cx) ** 2 + (y_grid - cy) ** 2)
        max_dist = np.sqrt(cx ** 2 + cy ** 2)
        return dist > 0.6 * max_dist

    @lru_cache(maxsize=16)
    def _fft_sector_masks(shape, num_sectors=8):
        h, w = shape
        cy, cx = h // 2, w // 2
        y_grid, x_grid = np.ogrid[:h, :w]
        angles = np.arctan2(y_grid - cy, x_grid - cx)
        masks = []
        for i in range(num_sectors):
            lo = -np.pi + i * (2.0 * np.pi / num_sectors)
            hi = lo + (2.0 * np.pi / num_sectors)
            masks.append((angles >= lo) & (angles < hi))
        return tuple(masks)

    def analyze_laplacian(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        variance = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        log_variance = float(np.log10(variance + 1.0))
        return float(np.clip(_soft_two_tailed_score(log_variance, np.log10(81), np.log10(551), 0.25), 0.0, 1.0))

    def analyze_sobel(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        magnitude = np.sqrt(
            cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3) ** 2
            + cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3) ** 2
        )
        mean, std = float(magnitude.mean()), float(magnitude.std())
        ratio = float((magnitude > mean + 2.0 * std).sum() / magnitude.size)
        return float(np.clip((ratio - 0.02) / 0.08, 0.0, 1.0))

    def analyze_dft(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
        dft = cv2.dft(gray, flags=cv2.DFT_COMPLEX_OUTPUT)
        magnitude = np.abs(np.fft.fftshift(dft[:, :, 0] + 1j * dft[:, :, 1]))
        mask = _high_frequency_mask(magnitude.shape)
        ratio = float(magnitude[mask].sum() / (magnitude.sum() + 1e-9))
        return float(np.clip((ratio - 0.10) / 0.15, 0.0, 1.0))

    def analyze_fft(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
        magnitude = np.abs(np.fft.fftshift(np.fft.fft2(gray)))
        sector_values = [float(magnitude[mask].mean()) for mask in _fft_sector_masks(magnitude.shape)]
        sectors = np.array(sector_values, dtype=np.float32)
        return float(np.clip(sectors.std() / (sectors.mean() + 1e-9), 0.0, 1.0))

    def analyze_hist(bgr):
        h, w = bgr.shape[:2]
        y0, y1, x0, x1 = _center_crop_region(bgr)
        face = bgr[y0:y1, x0:x1]
        bg_mask = np.ones((h, w), dtype=np.uint8)
        bg_mask[y0:y1, x0:x1] = 0
        if int(bg_mask.sum()) < 100:
            return 0.0
        distances = []
        for ch in range(3):
            h_face = cv2.calcHist([face], [ch], None, [64], [0, 256])
            h_bg = cv2.calcHist([bgr], [ch], bg_mask, [64], [0, 256])
            cv2.normalize(h_face, h_face)
            cv2.normalize(h_bg, h_bg)
            distances.append(float(cv2.compareHist(h_face, h_bg, cv2.HISTCMP_BHATTACHARYYA)))
        return float(_soft_two_tailed_score(float(np.mean(distances)), 0.20, 0.38, 0.22))

    def analyze_hsv_skin(bgr):
        hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, np.array([0, 20, 70], np.uint8), np.array([25, 255, 255], np.uint8))
        pixels = hsv[:, :, 1][mask > 0].astype(np.float32)
        if pixels.size < 100:
            return 0.0
        mean_sat = float(pixels.mean())
        cv_sat = float(pixels.std()) / (mean_sat + 1e-9)
        sat_span = (float(np.percentile(pixels, 90)) - float(np.percentile(pixels, 10))) / (mean_sat + 1e-9)
        cv_score = _soft_two_tailed_score(cv_sat, 0.30, 0.52, 0.20)
        span_score = _soft_two_tailed_score(sat_span, 0.75, 1.15, 0.35)
        return float(np.clip(0.60 * cv_score + 0.40 * span_score, 0.0, 1.0))

    def analyze_noise_residual(bgr):
        denoised = cv2.fastNlMeansDenoisingColored(
            bgr,
            None,
            h=3,
            hColor=3,
            templateWindowSize=7,
            searchWindowSize=21,
        )
        flat = (bgr.astype(np.float32) - denoised.astype(np.float32)).ravel()
        mean_r = float(flat.mean())
        std_r = float(flat.std()) + 1e-9
        kurtosis = float(np.mean(((flat - mean_r) / std_r) ** 4) - 3.0)
        return float(np.clip(abs(kurtosis) / 4.0, 0.0, 1.0))

    def analyze_jpeg_blocking(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
        h, w = gray.shape
        if h < 24 or w < 24:
            return 0.0
        diff_v = np.abs(np.diff(gray, axis=1))
        diff_h = np.abs(np.diff(gray, axis=0))
        v_boundary_mask = (np.arange(diff_v.shape[1]) % 8) == 7
        h_boundary_mask = (np.arange(diff_h.shape[0]) % 8) == 7
        if not np.any(v_boundary_mask) or not np.any(h_boundary_mask):
            return 0.0
        boundary_energy = float(diff_v[:, v_boundary_mask].mean() + diff_h[h_boundary_mask, :].mean())
        interior_energy = float(diff_v[:, ~v_boundary_mask].mean() + diff_h[~h_boundary_mask, :].mean()) + 1e-9
        return float(_soft_two_tailed_score(boundary_energy / interior_energy, 0.95, 1.18, 0.40))

    def analyze_edge_ringing(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        gray_f = gray.astype(np.float32)
        high_pass = np.abs(gray_f - cv2.GaussianBlur(gray_f, (0, 0), 1.2))
        edges = cv2.Canny(gray, 80, 180)
        if int(edges.sum()) == 0:
            return 0.0
        inner = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1) > 0
        outer = cv2.dilate(edges, np.ones((7, 7), np.uint8), iterations=1) > 0
        ring = outer & (~inner)
        if ring.sum() < 50 or inner.sum() < 50:
            return 0.0
        ratio = float(high_pass[ring].mean()) / (float(high_pass[inner].mean()) + 1e-9)
        return float(_soft_two_tailed_score(ratio, 0.20, 0.55, 0.45))

    def analyze_texture_inconsistency(bgr):
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
        mean = cv2.GaussianBlur(gray, (0, 0), 3.0)
        local_var = np.clip(cv2.GaussianBlur(gray * gray, (0, 0), 3.0) - (mean * mean), 0.0, None)
        y0, y1, x0, x1 = _center_crop_region(bgr)
        face_var = float(local_var[y0:y1, x0:x1].mean())
        bg_mask = np.ones(gray.shape, dtype=bool)
        bg_mask[y0:y1, x0:x1] = False
        ratio = face_var / (float(local_var[bg_mask].mean()) + 1e-9)
        return float(_soft_two_tailed_score(ratio, 0.65, 1.35, 0.75))

    ANALYZERS: Dict[str, AnalyzerFn] = {
        "laplacian": analyze_laplacian,
        "sobel": analyze_sobel,
        "dft": analyze_dft,
        "fft": analyze_fft,
        "hist": analyze_hist,
        "hsv_skin": analyze_hsv_skin,
        "noise_residual": analyze_noise_residual,
        "jpeg_blocking": analyze_jpeg_blocking,
        "edge_ringing": analyze_edge_ringing,
        "texture_inconsistency": analyze_texture_inconsistency,
    }

    DEFAULT_WEIGHTS = {
        "laplacian": 0.08,
        "sobel": 0.08,
        "dft": 0.14,
        "fft": 0.14,
        "hist": 0.10,
        "hsv_skin": 0.08,
        "noise_residual": 0.10,
        "jpeg_blocking": 0.10,
        "edge_ringing": 0.09,
        "texture_inconsistency": 0.09,
    }

    ANALYZER_PROFILES = {
        "fast": (
            "laplacian",
            "sobel",
            "dft",
            "hsv_skin",
            "jpeg_blocking",
            "edge_ringing",
            "texture_inconsistency",
        ),
        "balanced": (
            "laplacian",
            "sobel",
            "dft",
            "hist",
            "hsv_skin",
            "jpeg_blocking",
            "edge_ringing",
            "texture_inconsistency",
        ),
        "full": tuple(ANALYZERS.keys()),
    }

    PROFILE_VARIANT_WEIGHTS = {
        "fast": {"original": 0.78, "clahe": 0.22},
        "balanced": {"original": 0.65, "clahe": 0.20, "sharpened": 0.15},
        "full": {"original": 0.55, "denoised": 0.15, "sharpened": 0.15, "clahe": 0.15},
    }

    PROFILE_MAX_IMAGE_SIDE = {"fast": 512, "balanced": 640, "full": 768}

    def resolve_analysis_profile(profile):
        normalized = str(profile or DEFAULT_ANALYSIS_PROFILE).strip().lower()
        return normalized if normalized in ANALYZER_PROFILES else DEFAULT_ANALYSIS_PROFILE

    def get_profile_feature_names(profile):
        return ANALYZER_PROFILES[resolve_analysis_profile(profile)]

    def get_profile_variant_weights(profile):
        return dict(PROFILE_VARIANT_WEIGHTS[resolve_analysis_profile(profile)])

    def get_profile_max_image_side(profile):
        return int(PROFILE_MAX_IMAGE_SIDE[resolve_analysis_profile(profile)])

    def build_image_variants(bgr, variant_names=None):
        requested = [name for name in (variant_names or ["original", "denoised", "sharpened", "clahe"]) if name]
        variants = {}
        if "original" in requested:
            variants["original"] = bgr
        if "denoised" in requested:
            variants["denoised"] = cv2.bilateralFilter(bgr, d=7, sigmaColor=50, sigmaSpace=50)
        if "sharpened" in requested:
            blurred = cv2.GaussianBlur(bgr, (0, 0), 1.0)
            variants["sharpened"] = np.clip(cv2.addWeighted(bgr, 1.35, blurred, -0.35, 0), 0, 255).astype(np.uint8)
        if "clahe" in requested:
            lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
            l_channel, a_channel, b_channel = cv2.split(lab)
            l_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l_channel)
            variants["clahe"] = cv2.cvtColor(cv2.merge([l_eq, a_channel, b_channel]), cv2.COLOR_LAB2BGR)
        return variants

    def _safe(name, fn, bgr):
        try:
            return name, float(np.clip(fn(bgr), 0.0, 1.0))
        except Exception:
            return name, 0.0

    def run_all_analyzers(bgr, parallel=True, max_workers=None, feature_names=None):
        selected = {name: ANALYZERS[name] for name in (feature_names or ANALYZERS.keys()) if name in ANALYZERS}
        if not parallel:
            return {name: _safe(name, fn, bgr)[1] for name, fn in selected.items()}
        workers = max_workers or max(2, min(len(selected), os.cpu_count() or 4))
        out = {}
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = {ex.submit(_safe, name, fn, bgr): name for name, fn in selected.items()}
            for fut in _as_completed(futures):
                name, value = fut.result()
                out[name] = value
        return out

    def run_multiview_analyzers(
        bgr,
        variant_weights=None,
        parallel=True,
        max_workers=None,
        feature_names=None,
        max_image_side=None,
    ):
        profile_weights = variant_weights or get_profile_variant_weights(DEFAULT_ANALYSIS_PROFILE)
        resized = _resize_for_analysis(bgr, max_image_side)
        variant_names = tuple(profile_weights.keys())
        variants = build_image_variants(resized, variant_names=variant_names)
        total_weight = sum(float(profile_weights.get(name, 0.0)) for name in variants)
        if total_weight <= 0.0:
            total_weight = float(len(variants))
            profile_weights = {name: 1.0 for name in variants}

        per_variant = {}
        workers = max_workers or max(2, min(len(variants), os.cpu_count() or 4))
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futures = {
                ex.submit(run_all_analyzers, image, False, None, feature_names): name
                for name, image in variants.items()
            }
            for fut in _as_completed(futures):
                per_variant[futures[fut]] = fut.result()

        feature_order = [name for name in (feature_names or ANALYZERS.keys()) if name in ANALYZERS]
        aggregated = {}
        for feature in feature_order:
            weighted_sum = sum(float(profile_weights.get(name, 0.0)) * per_variant[name].get(feature, 0.0) for name in variants)
            aggregated[feature] = float(weighted_sum / total_weight) if total_weight > 0 else 0.0
        return aggregated

PROFILE_NAME = resolve_analysis_profile(ANALYSIS_PROFILE)
PROFILE_FEATURE_ORDER = list(get_profile_feature_names(PROFILE_NAME))
PROFILE_VARIANT_WEIGHTS = get_profile_variant_weights(PROFILE_NAME)
PROFILE_MAX_IMAGE_SIDE = ANALYSIS_MAX_SIDE or get_profile_max_image_side(PROFILE_NAME)

print(f"Notebook profile      : {PROFILE_NAME}")
print(f"Profile features      : {PROFILE_FEATURE_ORDER}")
print(f"Profile variant names : {list(PROFILE_VARIANT_WEIGHTS.keys())}")
print(f"Profile max image side: {PROFILE_MAX_IMAGE_SIDE}")

## Feature Extraction

Each image is processed through the `fast` deterministic profile before training.  
That means `original` + `clahe` variants, the reduced feature set, and a resize cap at 512px on the longest side. The extracted vectors are cached per profile so reruns do not recompute stale full-profile features.

In [ ]:
FEATURE_ORDER = list(PROFILE_FEATURE_ORDER)

def extract_one(path: str) -> np.ndarray | None:
    bgr = cv2.imread(path)
    if bgr is None:
        return None
    scores = run_multiview_analyzers(
        bgr,
        variant_weights=PROFILE_VARIANT_WEIGHTS,
        parallel=False,
        feature_names=FEATURE_ORDER,
        max_image_side=PROFILE_MAX_IMAGE_SIDE,
    )
    return np.array([float(scores.get(f, 0.0)) for f in FEATURE_ORDER], dtype=np.float32)


def extract_all(df: pd.DataFrame, cache_path: Path) -> tuple[np.ndarray, np.ndarray]:
    if cache_path.exists():
        data = np.load(cache_path)
        cached_feature_order = data["feature_order"].tolist() if "feature_order" in data.files else []
        cached_profile = str(data["profile"].item()) if "profile" in data.files else None
        cached_max_side = int(data["max_image_side"].item()) if "max_image_side" in data.files else None

        if (
            cached_feature_order == FEATURE_ORDER
            and cached_profile == PROFILE_NAME
            and cached_max_side == PROFILE_MAX_IMAGE_SIDE
        ):
            X, y = data["X"], data["y"]
            print(f"Loaded cached features from {cache_path}  (shape: {X.shape})")
            return X, y

        print("Existing cache does not match the requested fast-profile settings. Rebuilding cache.")

    paths = df["path"].tolist()
    labels = df["label"].tolist()
    vectors: list[np.ndarray | None] = [None] * len(paths)
    valid_flags = [False] * len(paths)
    idx_map = {i: p for i, p in enumerate(paths)}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_one, p): i for i, p in enumerate(paths)}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting fast-profile features", unit="img"):
            i = futures[future]
            try:
                vec = future.result()
                if vec is not None:
                    vectors[i] = vec
                    valid_flags[i] = True
            except Exception as exc:
                print(f"  Error on {Path(idx_map[i]).name}: {exc}")

    keep_idx = [i for i, ok in enumerate(valid_flags) if ok]
    if not keep_idx:
        raise RuntimeError("Feature extraction produced no valid samples.")

    X = np.vstack([vectors[i] for i in keep_idx]).astype(np.float32)
    y = np.array([labels[i] for i in keep_idx], dtype=np.float32)

    np.savez_compressed(
        cache_path,
        X=X,
        y=y,
        feature_order=np.array(FEATURE_ORDER),
        profile=np.array(PROFILE_NAME),
        max_image_side=np.array(PROFILE_MAX_IMAGE_SIDE),
    )
    print(f"Saved feature cache → {cache_path}")
    print(f"Dataset shape: X={X.shape}  y={y.shape}  ({int((y == 0).sum())} real, {int((y == 1).sum())} fake)")
    return X, y


print(f"Training notebook profile: {PROFILE_NAME}  |  features={len(FEATURE_ORDER)}")
X, y = extract_all(df, CACHE_PATH)

In [ ]:
import math
import matplotlib.pyplot as plt

num_features = len(FEATURE_ORDER)
num_cols = 3
num_rows = math.ceil(num_features / num_cols)
fig, axes = plt.subplots(num_rows, num_cols, figsize=(6 * num_cols, 4 * num_rows))
axes = np.atleast_1d(axes).flatten()

real_mask = y == 0
fake_mask = y == 1
colors = {"Real": "#4CAF50", "Fake": "#F44336"}

for ax, fname in zip(axes, FEATURE_ORDER):
    idx = FEATURE_ORDER.index(fname)
    real_vals = X[real_mask, idx]
    fake_vals = X[fake_mask, idx]
    parts = ax.violinplot([real_vals, fake_vals], positions=[0, 1], showmedians=True)
    for body, color in zip(parts["bodies"], [colors["Real"], colors["Fake"]]):
        body.set_facecolor(color)
        body.set_alpha(0.75)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Real", "Fake"])
    ax.set_title(fname, fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)

for ax in axes[num_features:]:
    ax.axis("off")

fig.suptitle(f"Feature Score Distributions — Real vs Fake ({PROFILE_NAME} profile)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Training the Calibration Model

Trains a logistic regression classifier on the extracted `fast`-profile deterministic feature vectors, then selects the decision threshold that maximises F1 on the held-out validation split.  
The saved `deterministic_calibration.json` includes the profile metadata (`profile`, `variant_weights`, `max_image_side`) so you can keep serving aligned with the same feature pipeline.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_recall_curve,
)

# --- Split ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VALIDATION_FRACTION, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}  Val: {len(X_val)}")

# --- Normalise ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)

# --- Train ---
clf = LogisticRegression(
    penalty="l2",
    C=1.0 / (1e-3 * len(X_train)),   # equivalent L2 strength
    max_iter=EPOCHS,
    solver="lbfgs",
    random_state=42,
)
clf.fit(X_train_s, y_train)

# --- Select threshold ---
val_probs = clf.predict_proba(X_val_s)[:, 1]
thresholds = np.linspace(0.10, 0.90, 81)
best_thresh, best_f1 = 0.5, 0.0
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    f = f1_score(y_val, preds, zero_division=0)
    if f > best_f1:
        best_f1, best_thresh = f, float(t)

print(f"Best threshold: {best_thresh:.2f}  (val F1 = {best_f1:.4f})")

# --- Final metrics ---
y_pred = (val_probs >= best_thresh).astype(int)
print("\n" + classification_report(y_val, y_pred, target_names=["real", "fake"]))

In [ ]:
cm = confusion_matrix(y_val, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=["Pred Real", "Pred Fake"],
    yticklabels=["True Real", "True Fake"],
    title="Confusion Matrix (validation set)",
)
thresh_val = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > thresh_val else "black", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
weights = clf.coef_[0]                 # shape (n_features,)
sorted_idx = np.argsort(weights)       # ascending → most-negative first

fig, ax = plt.subplots(figsize=(9, 5))
colors_bar = ["#F44336" if w >= 0 else "#4CAF50" for w in weights[sorted_idx]]
ax.barh([FEATURE_ORDER[i] for i in sorted_idx], weights[sorted_idx], color=colors_bar)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Logistic regression coefficient (positive → pushes toward fake)")
ax.set_title("Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, roc_thresholds = roc_curve(y_val, val_probs)
roc_auc = auc(fpr, tpr)

# F1 over threshold sweep
f1_scores_thresh = []
for t in thresholds:
    preds_t = (val_probs >= t).astype(int)
    f1_scores_thresh.append(f1_score(y_val, preds_t, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, color="#2196F3", lw=2, label=f"AUC = {roc_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
axes[0].legend(loc="lower right")
axes[0].yaxis.grid(True, linestyle="--", alpha=0.4)

# F1 vs threshold
axes[1].plot(thresholds, f1_scores_thresh, color="#FF9800", lw=2)
axes[1].axvline(best_thresh, color="#F44336", linestyle="--",
                label=f"Best @{best_thresh:.2f}  F1={best_f1:.4f}")
axes[1].set(xlabel="Decision Threshold", ylabel="F1 Score",
            title="F1 Score vs Threshold Sweep", ylim=(0, 1.05))
axes[1].legend()
axes[1].yaxis.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
import json

calibration = {
    "feature_order": FEATURE_ORDER,
    "weights": clf.coef_[0].tolist(),
    "bias": float(clf.intercept_[0]),
    "means": scaler.mean_.tolist(),
    "scales": scaler.scale_.tolist(),
    "threshold": best_thresh,
    "profile": PROFILE_NAME,
    "variant_weights": {key: float(value) for key, value in PROFILE_VARIANT_WEIGHTS.items()},
    "max_image_side": int(PROFILE_MAX_IMAGE_SIDE),
    "metrics": {
        "validation_f1": float(best_f1),
        "roc_auc": float(roc_auc),
        "train_size": int(len(X_train)),
        "validation_size": int(len(X_val)),
    },
}

CALIB_OUTPUT = CALIB_PATH
with open(CALIB_OUTPUT, "w", encoding="utf-8") as fh:
    json.dump(calibration, fh, indent=2)

print(f"Saved calibration → {CALIB_OUTPUT}")
print(json.dumps({k: v for k, v in calibration.items() if k not in ("weights", "means", "scales")}, indent=2))

# Download in Colab
if IN_COLAB:
    from google.colab import files
    files.download(str(CALIB_OUTPUT))
    print("Download triggered — check your browser downloads.")

## Using the Calibration File

Copy `deterministic_calibration.json` to your server and set the environment variables before starting the API:

```bash
export DETERMINISTIC_CALIBRATION_PATH=/path/to/deterministic_calibration.json
export DETERMINISTIC_PROFILE=fast
export DETERMINISTIC_MAX_SIDE=512
export DEEPFAKE_USE_DETERMINISTIC=true
export DEEPFAKE_DETERMINISTIC_WEIGHT=0.35

uvicorn api:app --host 0.0.0.0 --port 8000
```

This notebook now trains the `fast` profile, so the serving path should use the same profile and resize cap when you deploy the resulting calibration file.